# Mt. Hood Ash Dispersal — Tephra2 Sensitivity Analysis

Which of the three swept eruption source parameters -- **plume height**, **eruption mass**, or **diffusion
coefficient** -- has the biggest effect on ash thickness? This notebook answers that with two simple,
standard sensitivity-analysis techniques, run fresh (not derived from `tephra2_sweep.ipynb`'s or
`tephra2_grid_sweep.ipynb`'s output):

1. **One-at-a-time (OAT)**: vary a single parameter across its full range while holding the other two fixed
   at a baseline value, and see how much the output moves. Repeated for each parameter separately, this
   directly ranks the three by how much they alone move ash thickness.
2. **Pairwise**: vary two parameters together over a 2D grid (holding the third fixed at baseline), producing
   a heatmap that shows both parameters' joint effect and any interaction between them.

This is intentionally a **standalone, toggle-driven notebook** -- Section 2 controls which tests actually run,
so you can enable just one OAT parameter, just the pairwise test, or everything, without needing to touch the
rest of the notebook. Because it only evaluates the 3 POIs (not a spatial grid), every test here is cheap --
tens to a couple hundred Tephra2 calls, not thousands -- so it's meant to be quick to re-run with different
settings, unlike the full parameter sweeps.

### Volcano context
Mt. Hood is a low-explosivity, dome-collapse stratovolcano. Its eruptive history is dominated by dacite dome
growth, pyroclastic flows, and lahars, with comparatively little pumiceous ash. This constrains the physically
realistic parameter ranges used below, particularly plume height — this is **not** a Plinian-eruption parameter
sweep.

### A note on rigor
OAT and pairwise sweeps are simple, interpretable, and cheap, but they only explore variation *around one
baseline point* and can't detect interactions among all three parameters at once the way a full factorial
sweep or a proper global sensitivity method (e.g. Sobol, Morris -- see Scott et al. 2025's Tephra2 global
sensitivity analysis) would. Treat the ranking here as a useful first-pass answer, not a substitute for that
kind of analysis if the thesis needs one.

## Section 1 — Setup

Vent location, elevation, fixed ESPs, the 3-POI grid file, and the wind file -- unchanged from
`tephra2_sweep.ipynb`.

In [ ]:
import os
import sys
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import utm
import cdsapi
import netCDF4

sys.path.insert(0, "/home/jovyan/shared/Libraries/")
import victor
import rioxarray as rxr

TEPHRA2_BIN = "/home/jovyan/tephra2/tephra2_2020"


In [ ]:
# Mt. Hood vent coordinates
vent_latitude, vent_longitude = 45.22, -121.44

converted = utm.from_latlon(vent_latitude, vent_longitude)
vent_easting = converted[0]
vent_northing = converted[1]
utm_zone_number = converted[2]
utm_zone_letter = converted[3]

dem_name = victor.download_dem(
    vent_latitude + 1, vent_latitude - 1, vent_longitude + 1, vent_longitude - 1,
    "tiff", "SRTMGL3", filename="tephra2.tiff", api_key='73d3f5f5000048606a3c15ff635bde63')

dem = rxr.open_rasterio(dem_name)
vent_elevation = dem.sel(x=vent_longitude, y=vent_latitude, method="nearest").values[0]

print(f"Vent UTM: {vent_easting:.0f} E, {vent_northing:.0f} N (zone {utm_zone_number}{utm_zone_letter})")
print(f"Vent elevation: {vent_elevation:.0f} m")


In [ ]:
median_grain = 1
std_grain = 1

falltime_thresh = 0
plume_model = 0
part_steps = 0
lithic_density = 0
col_steps = 0
max_grain = 0
alpha = 0
beta = 0
min_grain = 0
pumice_density = 0
eddy_const = 0


In [ ]:
elevation = 1000  # meters, fixed (flat-topography assumption)

poi_names = ["Rhododendron", "Parkdale", "Govt. Camp"]
poi_locations = np.array([
    [45.329563, -121.911191],
    [45.519839, -121.596742],
    [45.1808, -121.4509],
])
poi_utm = utm.from_latlon(poi_locations[:, 0], poi_locations[:, 1])
poi_easting = [round(poi_utm[0][i]) for i in range(len(poi_names))]
poi_northing = [round(poi_utm[1][i]) for i in range(len(poi_names))]

with open("volcano_cone.grid", "w") as output_file:
    for e, n in zip(poi_easting, poi_northing):
        print(e, n, elevation, file=output_file)

print("Wrote volcano_cone.grid (3 points)")


In [ ]:
months = ["10"]
years = ["2024"]
days = ["23"]
hours = ["17:00"]

pressures = ['1', '2', '3', '5', '7', '10', '20', '30', '50', '70',
             '100', '125', '150', '175', '200', '225', '250', '300',
             '350', '400', '450', '500', '550', '600', '650', '700',
             '750', '775', '800', '825', '850', '875', '900', '925',
             '950', '975', '1000']

north = round(vent_latitude * 4) / 4
south = north + 0.1
east = round(vent_longitude * 4) / 4
west = east - 0.1

wind_client = cdsapi.Client()
dataset = "reanalysis-era5-pressure-levels"
request = {
    "product_type": ["reanalysis"],
    "data_format": "netcdf",
    "variable": ["geopotential", "u_component_of_wind", "v_component_of_wind"],
    "pressure_level": pressures,
    "year": years,
    "month": months,
    "day": days,
    "time": hours,
    "download_format": "unarchived",
    "area": [north, west, south, east],
}
wind_client.retrieve(dataset, request, "download.nc")


In [ ]:
wind = netCDF4.Dataset("download.nc")

uwnd = wind["u"][0, :, 0, 0]
vwnd = wind["v"][0, :, 0, 0]

speed = np.sqrt(uwnd**2 + vwnd**2)
direction = -180 / np.pi * np.arctan(vwnd / uwnd)
for d in range(len(direction)):
    if uwnd[d] > 0:
        direction[d] += 90
    else:
        direction[d] += 270

hgt = wind["z"][0, :, 0, 0] / 9.80665

speed = speed[::-1]
direction = direction[::-1]
hgt = hgt[::-1]

with open("my_wind.dat", "w") as wind_file:
    for level in range(wind["pressure_level"].shape[0]):
        wind_file.write(f"{hgt[level]} {speed[level]} {direction[level]}\n")

print("Wrote my_wind.dat")


## Section 2 — Configuration

**Baseline** is the fixed value each parameter holds while the *other two* are being varied -- the point all
the OAT sweeps radiate out from. It's the geometric mean of each parameter's range for eruption mass and
diffusion coefficient (appropriate for values spanning orders of magnitude) and the linear mean for plume
height (its range spans less than one order of magnitude).

**`use_log_spacing`**: eruption mass (1e9-1e12) and diffusion coefficient (1e3-1e5) each span 2-3 orders of
magnitude; plume height (1,000-24,000) spans less than one. `True` samples mass/diffusion coefficient
log-spaced (`np.geomspace`) -- standard practice for parameters spanning orders of magnitude, and avoids
under-sampling their lower end -- while plume height stays linear either way. `False` makes all three linear
(`np.linspace`), matching `tephra2_sweep.ipynb`/`tephra2_grid_sweep.ipynb`'s convention exactly. Toggle and
re-run to compare both.

Toggle which tests actually run in the cells below.

In [ ]:
ph_start, ph_end = 1_000, 24_000      # plume height (m asl)
em_start, em_end = 1e9, 1e12          # eruption mass (kg)
dc_start, dc_end = 1e3, 1e5           # diffusion coefficient (m^2/s)

baseline_plume_height = (ph_start + ph_end) / 2
baseline_eruption_mass = np.sqrt(em_start * em_end)
baseline_diffusion_coef = np.sqrt(dc_start * dc_end)

print(f"Baseline: plume_height={baseline_plume_height:.0f} m, "
      f"eruption_mass={baseline_eruption_mass:.3g} kg, diffusion_coef={baseline_diffusion_coef:.3g} m^2/s")

use_log_spacing = True

n_steps_oat = 20        # points per parameter for the OAT sweeps
n_steps_pairwise = 15   # points per axis for the pairwise sweep (n_steps_pairwise**2 total runs)

run_oat_plume_height    = True
run_oat_eruption_mass   = True
run_oat_diffusion_coef  = True
run_pairwise            = True

# Which two parameters the pairwise test varies (the third is held at baseline). One of:
# 'plume_height', 'eruption_mass', 'diffusion_coef'
pairwise_x = 'plume_height'
pairwise_y = 'eruption_mass'


## Section 3 — Run a single Tephra2 scenario

Shared helper used by every test below: writes a config file for one `(plume_height, eruption_mass,
diffusion_coef)` combination, runs Tephra2 against the 3-POI grid, and returns each POI's ash thickness as a
dict. Same config-writing logic as `tephra2_sweep.ipynb` (`row` rebuilt from scratch per call,
`DIFFUSION_COEFFICIENT` never conditionally overwritten).

In [ ]:
def run_scenario(plume_height, eruption_mass, diffusion_coef):
    """Run Tephra2 for one parameter combination; return {poi_name: mass_kg_m2}."""
    row = []
    row.append('VENT_EASTING ' + str(vent_easting))
    row.append('VENT_NORTHING ' + str(vent_northing))
    row.append('VENT_ELEVATION ' + str(vent_elevation))
    row.append('PLUME_HEIGHT ' + str(plume_height))
    row.append('ERUPTION_MASS ' + str(eruption_mass))
    row.append('MEDIAN_GRAINSIZE ' + str(median_grain))
    row.append('STD_GRAINSIZE ' + str(std_grain))
    row.append("".join(('FALL_TIME_THRESHOLD ', str(1000) if not falltime_thresh else str(falltime_thresh))))
    row.append("".join(('PLUME_MODEL ', str(2) if not plume_model else str(plume_model))))
    row.append("".join(('PART_STEPS ', str(100) if not part_steps else str(part_steps))))
    row.append("".join(('LITHIC_DENSITY ', str(2600.0) if not lithic_density else str(lithic_density))))
    row.append("".join(('COL_STEPS ', str(200) if not col_steps else str(col_steps))))
    row.append("".join(('MAX_GRAINSIZE ', str(-4) if not max_grain else str(max_grain))))
    row.append("".join(('ALPHA ', str(1) if not alpha else str(alpha))))
    row.append("".join(('BETA ', str(1) if not beta else str(beta))))
    row.append("".join(('MIN_GRAINSIZE ', str(4) if not min_grain else str(min_grain))))
    row.append("".join(('PUMICE_DENSITY ', str(1000.0) if not pumice_density else str(pumice_density))))
    row.append("".join(('EDDY_CONST ', str(0.04) if not eddy_const else str(eddy_const))))
    row.append('DIFFUSION_COEFFICIENT ' + str(diffusion_coef))

    with open("my_esps.conf", "w") as config_file:
        for element in row:
            config_file.write(element + "\n")

    csv_filename = "tephra2_temp.csv"
    result = subprocess.run(
        f'{TEPHRA2_BIN} my_esps.conf volcano_cone.grid my_wind.dat > {csv_filename} 2>/dev/null',
        shell=True)
    os.remove("my_esps.conf")

    if result.returncode != 0:
        print(f"tephra2 failed for PH={plume_height} EM={eruption_mass} DC={diffusion_coef}")
        os.remove(csv_filename)
        return {name: np.nan for name in poi_names}

    tephra_out = pd.read_csv(csv_filename, sep=r'\s+')
    tephra_out = tephra_out.rename(columns={'#EAST': 'easting', 'NORTH': 'northing', 'Kg/m^2': 'mass_kg_m2'})
    os.remove(csv_filename)

    results = {}
    for name, e, n in zip(poi_names, poi_easting, poi_northing):
        match = tephra_out[(tephra_out['easting'] == e) & (tephra_out['northing'] == n)]
        mass = match['mass_kg_m2'].iloc[0]
        results[name] = 0.0 if mass < 1e-300 else mass  # underflow -> exact zero
    return results


## Section 4 — One-at-a-time (OAT) sweeps

Each cell below varies one parameter across its full range while holding the other two at baseline, and
saves the result to its own CSV (`oat_plume_height.csv`, `oat_eruption_mass.csv`, `oat_diffusion_coef.csv`).

In [ ]:
def run_oat(param_name, values):
    rows = []
    for value in values:
        ph = value if param_name == 'plume_height' else baseline_plume_height
        em = value if param_name == 'eruption_mass' else baseline_eruption_mass
        dc = value if param_name == 'diffusion_coef' else baseline_diffusion_coef
        result = run_scenario(int(round(ph)), em, dc)
        rows.append({param_name: value, **result})

    df = pd.DataFrame(rows)
    df.to_csv(f"oat_{param_name}.csv", index=False)
    print(f"Wrote oat_{param_name}.csv ({len(df)} runs)")
    return df


In [ ]:
oat_results = {}

if run_oat_plume_height:
    values = np.linspace(ph_start, ph_end, n_steps_oat)
    oat_results['plume_height'] = run_oat('plume_height', values)


In [ ]:
if run_oat_eruption_mass:
    values = (np.geomspace(em_start, em_end, n_steps_oat) if use_log_spacing
              else np.linspace(em_start, em_end, n_steps_oat))
    oat_results['eruption_mass'] = run_oat('eruption_mass', values)


In [ ]:
if run_oat_diffusion_coef:
    values = (np.geomspace(dc_start, dc_end, n_steps_oat) if use_log_spacing
              else np.linspace(dc_start, dc_end, n_steps_oat))
    oat_results['diffusion_coef'] = run_oat('diffusion_coef', values)


## Section 5 — OAT Plots & Sensitivity Ranking

Per-parameter plots (ash thickness vs. the swept parameter, one line per POI), a combined overlay comparing
all tested parameters on one normalized axis, and a tornado diagram ranking parameters by how much they move
the output.

In [ ]:
PARAM_LABELS = {
    'plume_height': 'Plume height (m asl)',
    'eruption_mass': 'Eruption mass (kg)',
    'diffusion_coef': 'Diffusion coefficient (m²/s)',
}

def plot_oat(param_name):
    df = oat_results[param_name]
    fig, ax = plt.subplots(figsize=(8, 5))
    for name in poi_names:
        ax.plot(df[param_name], df[name], marker='o', markersize=3, label=name)

    ax.set_yscale('log')
    if param_name in ('eruption_mass', 'diffusion_coef') and use_log_spacing:
        ax.set_xscale('log')
    ax.set_xlabel(PARAM_LABELS[param_name])
    ax.set_ylabel('Ash thickness (kg/m²)')
    ax.set_title(f'OAT sensitivity: {PARAM_LABELS[param_name]}\n(other two parameters held at baseline)')
    ax.legend()
    fig.tight_layout()
    fig.savefig(f'oat_{param_name}.png', dpi=150, bbox_inches='tight')
    plt.show()


for param_name in oat_results:
    plot_oat(param_name)


In [ ]:
def sensitivity_score(df, param_name):
    """Average, across POIs, of log10(max/min) over the swept range -- orders of magnitude the
    output moves when this parameter alone is varied. Comparable across parameters regardless of
    their own units/scale, and robust to ash thickness's huge dynamic range."""
    scores = {}
    for name in poi_names:
        values = df[name].replace(0, np.nan).dropna()
        if len(values) < 2:
            scores[name] = 0.0
            continue
        scores[name] = np.log10(values.max() / values.min())
    scores['average'] = float(np.mean(list(scores.values())))
    return scores


sensitivity_summary = pd.DataFrame({
    param_name: sensitivity_score(df, param_name) for param_name, df in oat_results.items()
}).T
sensitivity_summary = sensitivity_summary.sort_values('average', ascending=False)
sensitivity_summary


In [ ]:
# Combined overlay: all tested parameters on one normalized x-axis (0 = low end of swept range,
# 1 = high end, by step index -- avoids needing to reconcile linear vs. log spacing across parameters),
# y-axis normalized to each parameter's own baseline-run value so relative sensitivity is comparable.
fig, ax = plt.subplots(figsize=(8, 5))
for param_name, df in oat_results.items():
    n = len(df)
    x_norm = np.linspace(0, 1, n)
    for name in poi_names:
        y = df[name].values
        baseline_idx = n // 2
        baseline_val = y[baseline_idx] if y[baseline_idx] > 0 else np.nanmedian(y[y > 0])
        y_norm = y / baseline_val if baseline_val and baseline_val > 0 else y
        ax.plot(x_norm, y_norm, alpha=0.5, lw=1,
                label=f'{PARAM_LABELS[param_name]} ({name})' if name == poi_names[0] else None,
                color=f'C{list(oat_results).index(param_name)}')

ax.set_yscale('log')
ax.set_xlabel('Fraction of swept range (low -> high)')
ax.set_ylabel('Ash thickness / mid-range value')
ax.set_title('OAT sensitivity overlay -- steeper/wider lines = more sensitive parameter')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig('oat_overlay.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Tornado diagram: horizontal bars spanning each parameter's min-max output range (log scale),
# sorted by range -- the classic sensitivity-ranking visualization.
fig, ax = plt.subplots(figsize=(8, 4))
tornado_rows = []
for param_name, df in oat_results.items():
    for name in poi_names:
        values = df[name].replace(0, np.nan).dropna()
        if len(values):
            tornado_rows.append({'parameter': PARAM_LABELS[param_name], 'location': name,
                                  'min': values.min(), 'max': values.max()})
tornado_df = pd.DataFrame(tornado_rows)
agg = tornado_df.groupby('parameter').agg(min=('min', 'min'), max=('max', 'max'))
agg = agg.loc[(agg['max'] / agg['min'].replace(0, np.nan)).sort_values(ascending=False).index]

y_pos = np.arange(len(agg))
ax.barh(y_pos, agg['max'] - agg['min'], left=agg['min'], color='tab:orange', alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(agg.index)
ax.set_xscale('log')
ax.set_xlabel('Ash thickness range across swept parameter (kg/m²), all POIs combined')
ax.set_title('Tornado diagram: which parameter moves ash thickness the most?')
fig.tight_layout()
fig.savefig('sensitivity_tornado.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 6 — Pairwise Comparison

Varies `pairwise_x` and `pairwise_y` together over a `n_steps_pairwise x n_steps_pairwise` grid, holding the
third parameter at baseline, and plots the result as a heatmap per POI -- shows both parameters' joint effect,
including any interaction (e.g. one parameter mattering more or less depending on the other's value) that OAT
alone can't reveal.

In [ ]:
def cell_edges(centers):
    """Quadrilateral edges for pcolormesh(shading='flat') pinned exactly to the data range --
    see tephra2_analysis.ipynb for why this matters with a small number of steps."""
    centers = np.asarray(centers, dtype=float)
    if len(centers) == 1:
        return np.array([centers[0] - 0.5, centers[0] + 0.5])
    midpoints = (centers[:-1] + centers[1:]) / 2
    return np.concatenate(([centers[0]], midpoints, [centers[-1]]))


PARAM_RANGES = {
    'plume_height': (ph_start, ph_end, False),
    'eruption_mass': (em_start, em_end, use_log_spacing),
    'diffusion_coef': (dc_start, dc_end, use_log_spacing),
}
PARAM_BASELINES = {
    'plume_height': baseline_plume_height,
    'eruption_mass': baseline_eruption_mass,
    'diffusion_coef': baseline_diffusion_coef,
}


def param_values(param_name, n):
    start, end, log = PARAM_RANGES[param_name]
    return np.geomspace(start, end, n) if log else np.linspace(start, end, n)


In [ ]:
if run_pairwise:
    third_param = [p for p in PARAM_RANGES if p not in (pairwise_x, pairwise_y)][0]
    x_values = param_values(pairwise_x, n_steps_pairwise)
    y_values = param_values(pairwise_y, n_steps_pairwise)

    rows = []
    for x_val in x_values:
        for y_val in y_values:
            params = {third_param: PARAM_BASELINES[third_param], pairwise_x: x_val, pairwise_y: y_val}
            ph = params['plume_height'] if 'plume_height' in params else PARAM_BASELINES['plume_height']
            em = params['eruption_mass'] if 'eruption_mass' in params else PARAM_BASELINES['eruption_mass']
            dc = params['diffusion_coef'] if 'diffusion_coef' in params else PARAM_BASELINES['diffusion_coef']
            result = run_scenario(int(round(ph)), em, dc)
            rows.append({pairwise_x: x_val, pairwise_y: y_val, **result})

    pairwise_df = pd.DataFrame(rows)
    pairwise_df.to_csv(f"pairwise_{pairwise_x}_{pairwise_y}.csv", index=False)
    print(f"Wrote pairwise_{pairwise_x}_{pairwise_y}.csv ({len(pairwise_df)} runs)")


In [ ]:
if run_pairwise:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, name in zip(axes, poi_names):
        pivot = pairwise_df.pivot_table(index=pairwise_y, columns=pairwise_x, values=name)

        positive_values = pivot.values[pivot.values > 0]
        norm = LogNorm(vmin=max(positive_values.min(), 1e-3), vmax=pivot.values.max()) if len(positive_values) else None

        x_edges = cell_edges(pivot.columns.values)
        y_edges = cell_edges(pivot.index.values)
        pcm = ax.pcolormesh(x_edges, y_edges, pivot.values, norm=norm, cmap='YlOrRd', shading='flat')
        fig.colorbar(pcm, ax=ax, label='Ash thickness (kg/m²)')
        ax.set_xlabel(PARAM_LABELS[pairwise_x])
        ax.set_ylabel(PARAM_LABELS[pairwise_y])
        if PARAM_RANGES[pairwise_x][2]:
            ax.set_xscale('log')
        if PARAM_RANGES[pairwise_y][2]:
            ax.set_yscale('log')
        ax.set_title(name)

    fig.suptitle(f'{PARAM_LABELS[pairwise_x]} vs. {PARAM_LABELS[pairwise_y]} '
                 f'({third_param} = {PARAM_BASELINES[third_param]:.3g})', y=1.03)
    fig.tight_layout()
    fig.savefig(f'pairwise_{pairwise_x}_{pairwise_y}.png', dpi=150, bbox_inches='tight')
    plt.show()
